# 🤖 Project FORESIGHT – Notebook 04: Model Training & Selection
**NorthBay Living | Demand & Inventory Intelligence**

Trains all 6 forecasting models using Rolling-Origin Cross Validation and selects the best model based on WAPE.


In [ ]:
import sys; sys.path.insert(0, '..')
import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import joblib
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from pathlib import Path
from src.forecasting import (
    rolling_origin_splits, prepare_features, MODEL_CONFIGS, SeasonalNaive,
    cross_validate_models, train_best_model, generate_forecast, FEATURE_IMPORTANCE
)
from src.evaluation import compute_all_metrics, compare_models, select_best_model

DATA   = Path('../data/processed')
MODELS = Path('../models')
LAYOUT = dict(paper_bgcolor='#1e293b', plot_bgcolor='#1e293b', font=dict(color='#94a3b8'))
print('Ready ✓')

In [ ]:
feat = pd.read_csv(DATA / 'features_engineered.csv', parse_dates=['date'], low_memory=False)
counts = feat.groupby('stock_code')['date'].count()
valid_skus = counts[counts >= 60].index
df = feat[feat['stock_code'].isin(valid_skus)].sort_values(['stock_code','date'])
print(f'Modelling dataset: {df.shape[0]:,} rows | {df["stock_code"].nunique():,} SKUs')

## 1. Rolling-Origin Cross Validation (All Models)

In [ ]:
print('Running Rolling-Origin CV for all models…')
print('(This may take several minutes)')
cv_results = cross_validate_models(df, n_folds=3, forecast_horizon=28)
print('\nCV Complete!')

## 2. Model Comparison

In [ ]:
comparison_df = compare_models(cv_results)
print(comparison_df.to_string(index=False))

metrics = ['WAPE','MAPE','MAE','RMSE']
available = [m for m in metrics if m in comparison_df.columns]
colors = ['#6366f1','#06b6d4','#f59e0b','#10b981','#f97316','#ef4444']

fig = go.Figure()
for i, (_, row) in enumerate(comparison_df.iterrows()):
    fig.add_trace(go.Bar(
        name=row.get('model',''),
        x=available,
        y=[row.get(m,0) for m in available],
        marker_color=colors[i % len(colors)],
        text=[f'{row.get(m,0):.2f}' for m in available],
        textposition='outside', textfont_color='white'
    ))
fig.update_layout(barmode='group', title='Model Performance Comparison (3-Fold Rolling-Origin CV)',
    height=480, **LAYOUT)
fig.show()

## 3. Best Model Selection

In [ ]:
best_model_name = select_best_model(cv_results, metric='WAPE')
print(f'\n🏆 Best model: {best_model_name}')
print(f'   WAPE: {cv_results[best_model_name]["WAPE"]:.2f}%')
print(f'   MAE:  {cv_results[best_model_name]["MAE"]:.4f}')
print(f'   RMSE: {cv_results[best_model_name]["RMSE"]:.4f}')

## 4. Train Final Model on Full Data

In [ ]:
best_model = train_best_model(df, best_model_name)
print(f'Final model trained and saved: models/model_{best_model_name}.joblib')

## 5. Feature Importance

In [ ]:
fi_file = MODELS / f'feature_importance_{best_model_name}.csv'
if fi_file.exists():
    fi = pd.read_csv(fi_file)
    top = fi.nlargest(25, 'importance').sort_values('importance')
    
    fig = go.Figure(go.Bar(
        x=top['importance'], y=top['feature'], orientation='h',
        marker=dict(color=top['importance'], colorscale=[[0,'#6366f1'],[1,'#06b6d4']]),
        text=top['importance'].round(4), textposition='outside', textfont_color='white'
    ))
    fig.update_layout(title=f'Top 25 Feature Importances – {best_model_name}',
        height=650, **LAYOUT)
    fig.show()
else:
    print('Feature importance file not found (SeasonalNaive selected?)')

## 6. Generate 28-Day Forecast

In [ ]:
forecast_df = generate_forecast(df, best_model, best_model_name, horizon=28)
print(f'Forecast shape: {forecast_df.shape}')
print(forecast_df.head(10))

## 7. Forecast Visualisation (Sample SKU)

In [ ]:
sample_sku = forecast_df['stock_code'].value_counts().index[0]
hist = df[df['stock_code'] == sample_sku].sort_values('date').tail(90)
fcast = forecast_df[forecast_df['stock_code'] == sample_sku].sort_values('date')

fig = go.Figure()
fig.add_trace(go.Scatter(x=hist['date'], y=hist['quantity'], name='Historical',
    line=dict(color='#06b6d4', width=2)))
fig.add_trace(go.Scatter(x=fcast['date'], y=fcast['forecast_quantity'], name='Forecast',
    line=dict(color='#f59e0b', width=2.5, dash='dash'),
    mode='lines+markers', marker=dict(size=6)))
# Confidence band
upper = fcast['forecast_quantity'] * 1.25
lower = fcast['forecast_quantity'] * 0.75
fig.add_trace(go.Scatter(
    x=pd.concat([fcast['date'], fcast['date'][::-1]]),
    y=pd.concat([upper, lower[::-1]]),
    fill='toself', fillcolor='rgba(245,158,11,0.1)',
    line=dict(color='rgba(0,0,0,0)'), name='±25% CI'
))
fig.add_vline(x=fcast['date'].min(), line_dash='dot', line_color='#94a3b8')
fig.update_layout(title=f'28-Day Forecast – {sample_sku}', height=420, **LAYOUT)
fig.show()